In [1]:
"""
═══════════════════════════════════════════════════════════════════
  Kaggle Notebook Multi-Version Search
  Cerca stringhe in TUTTE le versioni dei notebook specificati.
  Aree di ricerca: codice, log/output celle, input/dataset metadata.
  ➜ Eseguire come cella di un notebook Kaggle.
═══════════════════════════════════════════════════════════════════
"""
import os, json, time, hashlib, requests
from collections import defaultdict
from datetime import datetime

# ─── CONFIGURAZIONE ──────────────────────────────────────────────
KAGGLE_TOKEN = "KGAT_0df9fde0b5bddfd56db9f51a9814f319"

SEARCH_TERMS = [
    "frame_cache_all80",
    "cholec80-features",
    "tool_annotation",
    "tool_annotations",
]

NOTEBOOKS = [
    "christiancorvino/cholec80-mstcn-multitask",
    "christiancorvino3/cholec80-5-1-recognition-transformer-v2",
    "christiancorvino3/cholec80-mstcn-singletask-2",
    "christiancorvino/check-features-resnet50-finetuned",
    "christiancorvino/cholec80-mstcn-singletask-teammates-features",
    "christiancorvino/cholec80-mstcn-multitask-teammates-features",
    "giorgiachiaro/notebook-medical-robotics",
    "christiancorvino/cholec80-resnet50-finetuning",
    "christiancorvino/cholec80-5-1-recognition-transformer",
    "christiancorvino/cholec80-5-1-temporal-transformer",
    "christiancorvino/cholec80-mstcn-singletask",
]

MAX_VERSIONS = 60          # massimo versioni da controllare per notebook
DELAY        = 1.2         # secondi tra una chiamata API e l'altra
SAVE_PATH    = "/kaggle/working/search_results.json"

# ─── SETUP ───────────────────────────────────────────────────────
API = "https://www.kaggle.com/api/v1"
S   = requests.Session()
S.headers.update({"Authorization": f"Bearer {KAGGLE_TOKEN}"})

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def api_get(path, params=None):
    time.sleep(DELAY)
    try:
        r = S.get(f"{API}/{path.lstrip('/')}", params=params or {}, timeout=30)
        return r
    except Exception as e:
        log(f"  ❌ Errore rete: {e}")
        return None

# ─── FUNZIONI RICERCA ────────────────────────────────────────────
def find_terms(text, area):
    """Restituisce lista di hit [{area, term, line, context}]"""
    hits = []
    if not text:
        return hits
    for n, line in enumerate(text.split("\n"), 1):
        low = line.lower()
        for t in SEARCH_TERMS:
            if t.lower() in low:
                hits.append({"area": area, "term": t,
                             "line": n, "context": line.strip()[:200]})
    return hits

def notebook_text(source):
    """Ritorna (codice, log_output) come testo"""
    code, out = [], []
    if isinstance(source, str):
        try:
            nb = json.loads(source)
        except (json.JSONDecodeError, ValueError):
            return source, ""            # è uno script .py
    elif isinstance(source, dict):
        nb = source
    else:
        return str(source), ""

    for cell in nb.get("cells", []):
        src = cell.get("source", [])
        code.extend(src if isinstance(src, list) else [src])
        for o in cell.get("outputs", []):
            if "text" in o:
                t = o["text"]
                out.extend(t if isinstance(t, list) else [t])
            for mime, c in o.get("data", {}).items():
                if "text" in mime:
                    out.extend(c if isinstance(c, list) else [c])
    return "\n".join(code), "\n".join(out)

def get_max_version(owner, slug):
    """Trova il numero dell'ultima versione tramite kernels/list"""
    r = api_get("kernels/list", {"user": owner, "search": slug, "pageSize": 100})
    if r and r.ok:
        for k in r.json():
            ref = k.get("ref", "")
            if slug in ref:
                v = k.get("currentRunningVersion", None)
                if v:
                    return int(v)
    return None

# ─── VERIFICA TOKEN ──────────────────────────────────────────────
log("🔑 Verifica token Kaggle...")
test = api_get("kernels/list", {"user": "christiancorvino", "pageSize": 1})
if test is None or test.status_code == 401:
    log("❌ Token non valido o scaduto. Rigenera il KGAT su kaggle.com/settings.")
    raise SystemExit(1)
log("✅ Token valido\n")

# ─── LOOP PRINCIPALE ────────────────────────────────────────────
all_results = []

for nb_ref in NOTEBOOKS:
    owner, slug = nb_ref.split("/")
    log(f"{'═'*60}")
    log(f"📓  {nb_ref}")
    log(f"{'═'*60}")

    # L'API pubblica di Kaggle purtroppo NON supporta più il pull di versioni vecchie
    # Effettuiamo la ricerca sull'ultima versione disponibile.
    r = api_get(f"kernels/pull/{owner}/{slug}")

    if r is None or not r.ok:
        log(f"  ❌ Impossibile scaricare il notebook. Codice: {r.status_code if r else 'Timeout'}")
        continue

    data   = r.json()
    source = data.get("source", "")
    meta   = data.get("metadata", {})

    log(f"  📌 Scansione ultima versione")
    hits = []

    # A) CODICE + LOG/OUTPUT CELLE
    code_txt, log_txt = notebook_text(source)
    hits += find_terms(code_txt, "CODICE")
    hits += find_terms(log_txt,  "LOG/OUTPUT")

    # B) INPUT / DATASET (metadata)
    inputs = meta.get("dataset_sources", []) + meta.get("kernel_sources", [])
    hits += find_terms("\n".join(str(i) for i in inputs), "INPUT")

    # C) METADATA (fallback: cerca nel JSON intero dei metadati)
    meta_hits = find_terms(json.dumps(meta, ensure_ascii=False), "METADATA")
    existing = {(h["term"], h["context"]) for h in hits}
    hits += [h for h in meta_hits if (h["term"], h["context"]) not in existing]

    # D) OUTPUT FILES (nomi file prodotti)
    out_r = api_get(f"kernels/output/{owner}/{slug}")
    if out_r and out_r.ok:
        hits += find_terms(json.dumps(out_r.json(), ensure_ascii=False), "OUTPUT_FILES")

    # Report
    if hits:
        for h in hits:
            log(f"    ✅ [{h['area']}] '{h['term']}' — L{h['line']}: {h['context'][:120]}")
            all_results.append({"notebook": nb_ref, "version": "latest", **h})
    else:
        log(f"    ∅ nessun match nell'ultima versione")

# ─── RIEPILOGO ───────────────────────────────────────────────────
log(f"\n\n{'═'*60}")
log(f"📊  RIEPILOGO FINALE")
log(f"{'═'*60}\n")

if not all_results:
    log("Nessuna corrispondenza trovata in nessun notebook.")
else:
    by_term = defaultdict(list)
    for r in all_results:
        by_term[r["term"]].append(r)

    for t in SEARCH_TERMS:
        matches = by_term.get(t, [])
        log(f"🔍 '{t}' — {len(matches)} occorrenze totali:")
        if matches:
            # Raggruppa per notebook
            by_nb = defaultdict(list)
            for m in matches:
                by_nb[m["notebook"]].append(m)
            for nb, ms in by_nb.items():
                versions = sorted(set(m["version"] for m in ms))
                log(f"   📓 {nb}  (v{', v'.join(map(str, versions))})")
                for m in ms[:5]:  # mostra max 5 esempi
                    log(f"      [{m['area']}] L{m['line']}: {m['context'][:100]}")
                if len(ms) > 5:
                    log(f"      ... e altre {len(ms)-5} occorrenze")
        else:
            log(f"   (nessuna)")

log(f"\n📈 Totale: {len(all_results)} occorrenze trovate in {len(NOTEBOOKS)} notebook")

# Salva JSON
try:
    with open(SAVE_PATH, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    log(f"💾 Risultati salvati in {SAVE_PATH}")
except Exception:
    log("⚠ Impossibile salvare il file (se non sei su Kaggle, cambia SAVE_PATH)")

[15:29:43] 🔑 Verifica token Kaggle...
[15:29:44] ✅ Token valido

[15:29:44] ════════════════════════════════════════════════════════════
[15:29:44] 📓  christiancorvino/cholec80-mstcn-multitask
[15:29:44] ════════════════════════════════════════════════════════════
[15:29:46]   📌 Scansione ultima versione
[15:29:47]     ✅ [METADATA] 'cholec80-features' — L1: {"languageNullable": "python", "kernelTypeNullable": "notebook", "isPrivateNullable": true, "enableGpuNullable": true, "
[15:29:47] ════════════════════════════════════════════════════════════
[15:29:47] 📓  christiancorvino3/cholec80-5-1-recognition-transformer-v2
[15:29:47] ════════════════════════════════════════════════════════════
[15:29:49]   📌 Scansione ultima versione
[15:29:50]     ✅ [METADATA] 'cholec80-features' — L1: {"languageNullable": "python", "kernelTypeNullable": "notebook", "isPrivateNullable": true, "enableGpuNullable": true, "
[15:29:50] ════════════════════════════════════════════════════════════
[15:29:50] 📓  c